# Bab 1: Import Library & Setup

In [ ]:
# ======================================================================
# Bab 1 (VERSI FINAL REVISI 2): Instalasi, Restart, dan Setup
# PERHATIAN: SEL INI HARUS DIJALANKAN DUA KALI!
# ======================================================================

# --- Bagian 1: Instalasi & Restart (Hanya Efektif pada Run Pertama) ---
try:
    import mediapipe
    print("✅ Dependensi (NumPy & MediaPipe) tampaknya sudah OK.")
except ImportError:
    print("🔴 Dependensi tidak ditemukan atau perlu di-upgrade. Memulai instalasi...")
    !pip install "numpy>=2.0.0" mediapipe ultralytics -q
    print("✅ Instalasi selesai.")
    import os
    print("🔴 MERESTART RUNTIME... Tunggu hingga sesi restart, lalu JALANKAN SEL INI LAGI.")
    os.kill(os.getpid(), 9)

# --- Bagian 2: Import Semua Library (Efektif pada Run Kedua) ---
# Semua import diletakkan di bagian atas agar terdefinisi terlebih dahulu.
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder, Normalizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, Dropout,
    Conv2D, BatchNormalization, Activation, MaxPooling2D,
    Concatenate, Lambda, add
)
from functools import partial
import warnings
import requests
from io import BytesIO
from IPython.display import clear_output
import time
import traceback
from ultralytics import YOLO
import mediapipe as mp

# --- Bagian 3: Setup dan Inisialisasi (Efektif pada Run Kedua) ---
# Sekarang kita bisa menggunakan library yang sudah di-import.
warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print("\n--- Setup Selesai ---")
print(f"✅ NumPy version: {np.__version__}")
print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ MediaPipe version: {mp.__version__}")

# Inisialisasi solusi MediaPipe
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(min_detection_confidence=0.5)
print("✅ Detektor wajah MediaPipe berhasil diinisialisasi.")

✅ Dependensi (NumPy & MediaPipe) tampaknya sudah OK.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

--- Setup Selesai ---
✅ NumPy version: 2.0.2
✅ TensorFlow version: 2.18.0
✅ MediaPipe version: 0.10.14
✅ Detektor wajah MediaPipe berhasil diinisialisasi.


# Bab 2: Mount Google Drive


In [ ]:

# Pastikan ini dijalankan di awal sesi Google Colab
from google.colab import drive
drive.mount('/content/drive')

# Definisikan path k  e folder Output Files di Google Drive Anda
output_files_path = '/content/drive/MyDrive/Magang KP Model AI /Output Files Facenet 512'

os.makedirs(output_files_path, exist_ok=True) # Pastikan direktori ada

print(f"📂 Output files directory: {output_files_path}")

Mounted at /content/drive
📂 Output files directory: /content/drive/MyDrive/Magang KP Model AI /Output Files Facenet 512


# Bab 3: Re-definisi Model FaceNet (untuk memuat bobot yang sudah disimpan)

In [ ]:
# Kita perlu mendefinisikan ulang arsitektur model sebelum memuat bobotnya.
# Kode ini disalin dari bagian "1. Data Loading & Preprocessing" di notebook sebelumnya.

def scaling(x, scale):
    """Scaling function for residual connections"""
    return x * scale

def _generate_layer_name(name, branch_idx=None, prefix=None):
    """Generate layer names for the network"""
    if prefix is None:
        return None
    if branch_idx is None:
        return '_'.join((prefix, name))
    return '_'.join((prefix, 'Branch', str(branch_idx), name))

def conv2d_bn(x, filters, kernel_size, strides=1, padding='same',
              activation='relu', use_bias=False, name=None):
    """Convolution + Batch Normalization + Activation block"""
    x = Conv2D(filters, kernel_size, strides=strides, padding=padding,
               use_bias=use_bias, name=name)(x)

    if not use_bias:
        bn_axis = -1 # tf.keras.backend.image_data_format() == 'channels_last' secara default
        bn_name = _generate_layer_name('BatchNorm', prefix=name)
        x = BatchNormalization(axis=bn_axis, momentum=0.995, epsilon=0.001,
                              scale=False, name=bn_name)(x)

    if activation is not None:
        ac_name = _generate_layer_name('Activation', prefix=name)
        x = Activation(activation, name=ac_name)(x)

    return x

def _inception_resnet_block(x, scale, block_type, block_idx, activation='relu'):
    """Inception-ResNet block implementation"""
    channel_axis = -1 # tf.keras.backend.image_data_format() == 'channels_last' secara default

    if block_idx is None:
        prefix = None
    else:
        prefix = '_'.join((block_type, str(block_idx)))

    name_fmt = partial(_generate_layer_name, prefix=prefix)

    # Different block types
    if block_type == 'Block35':
        branch_0 = conv2d_bn(x, 32, 1, name=name_fmt('Conv2d_1x1', 0))
        branch_1 = conv2d_bn(x, 32, 1, name=name_fmt('Conv2d_0a_1x1', 1))
        branch_1 = conv2d_bn(branch_1, 32, 3, name=name_fmt('Conv2d_0b_3x3', 1))
        branch_2 = conv2d_bn(x, 32, 1, name=name_fmt('Conv2d_0a_1x1', 2))
        branch_2 = conv2d_bn(branch_2, 32, 3, name=name_fmt('Conv2d_0b_3x3', 2))
        branch_2 = conv2d_bn(branch_2, 32, 3, name=name_fmt('Conv2d_0c_3x3', 2))
        branches = [branch_0, branch_1, branch_2]
    elif block_type == 'Block17':
        branch_0 = conv2d_bn(x, 128, 1, name=name_fmt('Conv2d_1x1', 0))
        branch_1 = conv2d_bn(x, 128, 1, name=name_fmt('Conv2d_0a_1x1', 1))
        branch_1 = conv2d_bn(branch_1, 128, [1, 7], name=name_fmt('Conv2d_0b_1x7', 1))
        branch_1 = conv2d_bn(branch_1, 128, [7, 1], name=name_fmt('Conv2d_0c_7x1', 1))
        branches = [branch_0, branch_1]
    elif block_type == 'Block8':
        branch_0 = conv2d_bn(x, 192, 1, name=name_fmt('Conv2d_1x1', 0))
        branch_1 = conv2d_bn(x, 192, 1, name=name_fmt('Conv2d_0a_1x1', 1))
        branch_1 = conv2d_bn(branch_1, 192, [1, 3], name=name_fmt('Conv2d_0b_1x3', 1))
        branch_1 = conv2d_bn(branch_1, 192, [3, 1], name=name_fmt('Conv2d_0c_3x1', 1))
        branches = [branch_0, branch_1]
    else:
        raise ValueError(f'Unknown Inception-ResNet block type: {block_type}')

    mixed = Concatenate(axis=channel_axis, name=name_fmt('Concatenate'))(branches)

    # === FIX IS HERE ===
    # NEW, CORRECTED LINE: Get the number of filters from the KerasTensor's known shape.
    num_filters = x.shape[channel_axis]

    # Use the corrected variable in the next layer.
    up = conv2d_bn(mixed, num_filters, 1, activation=None, use_bias=True,
                   name=name_fmt('Conv2d_1x1'))
    # ====================

    up = Lambda(scaling, output_shape=lambda input_shape: input_shape,
                arguments={'scale': scale})(up)
    x = add([x, up])

    if activation is not None:
        x = Activation(activation, name=name_fmt('Activation'))(x)

    return x

def InceptionResNetV1(input_shape=(160, 160, 3), classes=128,
                      dropout_keep_prob=0.8, weights_path=None):
    """Create Inception-ResNet-V1 model for face recognition"""

    inputs = Input(shape=input_shape)
    x = conv2d_bn(inputs, 32, 3, strides=2, padding='valid', name='Conv2d_1a_3x3')
    x = conv2d_bn(x, 32, 3, padding='valid', name='Conv2d_2a_3x3')
    x = conv2d_bn(x, 64, 3, name='Conv2d_2b_3x3')
    x = MaxPooling2D(3, strides=2, name='MaxPool_3a_3x3')(x)
    x = conv2d_bn(x, 80, 1, padding='valid', name='Conv2d_3b_1x1')
    x = conv2d_bn(x, 192, 3, padding='valid', name='Conv2d_4a_3x3')
    x = conv2d_bn(x, 256, 3, strides=2, padding='valid', name='Conv2d_4b_3x3')

    # 5x Block35 (Inception-ResNet-A block)
    for block_idx in range(1, 6):
        x = _inception_resnet_block(x, scale=0.17, block_type='Block35', block_idx=block_idx)

    # Mixed 6a (Reduction-A block)
    channel_axis = -1 # tf.keras.backend.image_data_format() == 'channels_last' secara default
    name_fmt = partial(_generate_layer_name, prefix='Mixed_6a')

    branch_0 = conv2d_bn(x, 384, 3, strides=2, padding='valid', name=name_fmt('Conv2d_1a_3x3', 0))
    branch_1 = conv2d_bn(x, 192, 1, name=name_fmt('Conv2d_0a_1x1', 1))
    branch_1 = conv2d_bn(branch_1, 192, 3, name=name_fmt('Conv2d_0b_3x3', 1))
    branch_1 = conv2d_bn(branch_1, 256, 3, strides=2, padding='valid', name=name_fmt('Conv2d_1a_3x3', 1))
    branch_pool = MaxPooling2D(3, strides=2, padding='valid', name=name_fmt('MaxPool_1a_3x3', 2))(x)

    branches = [branch_0, branch_1, branch_pool]
    x = Concatenate(axis=channel_axis, name='Mixed_6a')(branches)

    # 10x Block17 (Inception-ResNet-B block)
    for block_idx in range(1, 11):
        x = _inception_resnet_block(x, scale=0.1, block_type='Block17', block_idx=block_idx)

    # Mixed 7a (Reduction-B block)
    name_fmt = partial(_generate_layer_name, prefix='Mixed_7a')

    branch_0 = conv2d_bn(x, 256, 1, name=name_fmt('Conv2d_0a_1x1', 0))
    branch_0 = conv2d_bn(branch_0, 384, 3, strides=2, padding='valid', name=name_fmt('Conv2d_1a_3x3', 0))
    branch_1 = conv2d_bn(x, 256, 1, name=name_fmt('Conv2d_0a_1x1', 1))
    branch_1 = conv2d_bn(branch_1, 256, 3, strides=2, padding='valid', name=name_fmt('Conv2d_1a_3x3', 1))
    branch_2 = conv2d_bn(x, 256, 1, name=name_fmt('Conv2d_0a_1x1', 2))
    branch_2 = conv2d_bn(branch_2, 256, 3, name=name_fmt('Conv2d_0b_3x3', 2))
    branch_2 = conv2d_bn(branch_2, 256, 3, strides=2, padding='valid', name=name_fmt('Conv2d_1a_3x3', 2))
    branch_pool = MaxPooling2D(3, strides=2, padding='valid', name=name_fmt('MaxPool_1a_3x3', 3))(x)

    branches = [branch_0, branch_1, branch_2, branch_pool]
    x = Concatenate(axis=channel_axis, name='Mixed_7a')(branches)

    # 5x Block8 (Inception-ResNet-C block)
    for block_idx in range(1, 6):
        x = _inception_resnet_block(x, scale=0.2, block_type='Block8', block_idx=block_idx)

    x = _inception_resnet_block(x, scale=1., activation=None, block_type='Block8', block_idx=6)

    # Classification block
    x = GlobalAveragePooling2D(name='AvgPool')(x)
    x = Dropout(1.0 - dropout_keep_prob, name='Dropout')(x)
    x = Dense(classes, use_bias=False, name='Bottleneck')(x)

    bn_name = _generate_layer_name('BatchNorm', prefix='Bottleneck')
    x = BatchNormalization(momentum=0.995, epsilon=0.001, scale=False, name=bn_name)(x)

    model = Model(inputs, x, name='inception_resnet_v1')

    if weights_path is not None:
        model.load_weights(weights_path)

    return model

# Inisialisasi model FaceNet dan muat bobot yang sudah disimpan
print("🔧 Initializing FaceNet-512 model...") # Ganti pesan untuk kejelasan
# === PERUBAHAN 1: Ganti 'classes' menjadi 512 ===
embeddings_generator = InceptionResNetV1(input_shape=(160, 160, 3), classes=512)
facenet_weights_path = os.path.join(output_files_path, 'facenet_model_weights.weights.h5')

if os.path.exists(facenet_weights_path):
    # === PERUBAHAN 2: Gunakan parameter tambahan untuk keamanan (opsional tapi disarankan) ===
    embeddings_generator.load_weights(facenet_weights_path)
    print("✅ FaceNet-512 model loaded successfully from saved weights.")
else:
    print(f"❌ FaceNet weights not found at {facenet_weights_path}.")
    print("Pastikan bagian pelatihan sebelumnya berjalan dengan sukses dan bobot disimpan.")
    # Opsional: Hentikan eksekusi jika bobot FaceNet tidak ditemukan
    # raise FileNotFoundError(f"FaceNet weights not found at {facenet_weights_path}")

🔧 Initializing FaceNet-512 model...
✅ FaceNet-512 model loaded successfully from saved weights.


#  Bab 4: Load Model & Data untuk Inference

## versi load model untuk json

In [ ]:
# ==================================================================
# Bab 4 (Versi JSON): Load Model & Data untuk Inference
# ==================================================================
import json

print("\n--- Memulai Bagian Inferensi ---")

# Muat model dan data yang disimpan dari pelatihan
print("⚙️ Memuat model yang diperlukan untuk inferensi...")
try:
    # 1. Muat Classifier (SVM atau Random Forest)
    best_classifier_path_svm = os.path.join(output_files_path, 'best_classifier_svm.pkl')
    best_classifier_path_rf = os.path.join(output_files_path, 'best_classifier_random_forest.pkl')

    if os.path.exists(best_classifier_path_svm):
        best_classifier = joblib.load(best_classifier_path_svm)
        print("✅ Classifier terbaik (SVM) berhasil dimuat.")
    elif os.path.exists(best_classifier_path_rf):
        best_classifier = joblib.load(best_classifier_path_rf)
        print("✅ Classifier terbaik (Random Forest) berhasil dimuat.")
    else:
        raise FileNotFoundError("Tidak ada classifier SVM maupun Random Forest yang ditemukan.")

    # 2. Muat LabelEncoder dan Normalizer
    label_encoder = joblib.load(os.path.join(output_files_path, 'label_encoder.pkl'))
    normalizer = joblib.load(os.path.join(output_files_path, 'normalizer.pkl'))
    print("✅ LabelEncoder dan Normalizer berhasil dimuat.")

    # 3. Muat Database Wajah dari File JSON
    database_json_path = os.path.join(output_files_path, 'face_database.json')
    if not os.path.exists(database_json_path):
        raise FileNotFoundError(f"Database JSON tidak ditemukan di: {database_json_path}. Jalankan skrip registrasi (Bab 10) terlebih dahulu.")

    with open(database_json_path, 'r') as f:
        face_database_list = json.load(f)
    print(f"✅ Membaca database dari '{os.path.basename(database_json_path)}'.")

    # 4. Ekstrak data dari struktur JSON ke format yang dibutuhkan (Numpy arrays)
    # Ini penting agar sisa kode kita yang menggunakan numpy tidak perlu diubah
    known_face_embeddings_raw = []
    known_face_labels = []

    for person_data in face_database_list:
        known_face_embeddings_raw.append(person_data["embedding"])
        known_face_labels.append(person_data["id"])

    # Konversi dari list kembali ke numpy array
    known_face_embeddings_raw = np.array(known_face_embeddings_raw)
    known_face_labels = np.array(known_face_labels)

    # Lakukan normalisasi pada embedding yang baru dimuat
    known_face_embeddings_norm = normalizer.transform(known_face_embeddings_raw)

    print(f"✅ Berhasil memuat dan memproses {len(known_face_labels)} wajah yang dikenal dari database JSON.")

except Exception as e:
    print(f"❌ Error saat memuat model/data yang disimpan: {e}")
    # exit() # Anda bisa uncomment ini jika ingin program berhenti saat ada error


--- Memulai Bagian Inferensi ---
⚙️ Memuat model yang diperlukan untuk inferensi...
✅ Classifier terbaik (SVM) berhasil dimuat.
✅ LabelEncoder dan Normalizer berhasil dimuat.
✅ Membaca database dari 'face_database.json'.
✅ Berhasil memuat dan memproses 3 wajah yang dikenal dari database JSON.


# Bab 6: Fungsi Preprocessing & Pengenalan Wajah

In [ ]:
!pip install scipy

In [ ]:
# Bab 6.1: Fungsi untuk Memproses File Video (VERSI MEDIAPIPE)
# =========================================================================
# FUNGSI INFERENSI VIDEO (VERSI OBJECT TRACKING LENGKAP DENGAN TEKS)
# =========================================================================
# Pastikan Anda sudah punya 'scipy'. Jika belum, jalankan: !pip install scipy
from scipy.spatial import distance as dist

def process_video_file(input_video_path, output_video_path):
    print(f"🎬 Memproses video dengan Object Tracking: {input_video_path}")

    cap = cv2.VideoCapture(input_video_path)
    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))
    fps, total_frames = int(cap.get(5)), int(cap.get(7))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    # Inisialisasi untuk Tracking
    tracked_faces = {}
    next_object_id = 0
    RECOGNITION_INTERVAL = 15
    MAX_DISAPPEARED_FRAMES = 10
    MAX_DISTANCE = 100 # Jarak piksel maksimum untuk dianggap objek yang sama

    for frame_idx in tqdm(range(total_frames), desc="Memproses Frame"):
        ret, frame = cap.read()
        if not ret: break

        annotated_frame = frame.copy()
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Tahap 1: Deteksi semua wajah di frame ini
        current_detections_bbox = []
        results = face_detector.process(rgb_frame)
        if results.detections:
            for detection in results.detections:
                bboxC = detection.location_data.relative_bounding_box
                ih, iw, _ = frame.shape
                x, y, w, h = int(bboxC.xmin * iw), int(bboxC.ymin * ih), int(bboxC.width * iw), int(bboxC.height * ih)
                current_detections_bbox.append((x, y, w, h))

        # Tahap 2: Update tracker
        # ... (Logika matching kompleks, disederhanakan untuk kejelasan)
        # Untuk kesederhanaan, kita gunakan logika yang mirip dengan sebelumnya, tapi lebih bersih

        # Buat daftar ID yang belum dicocokkan dari frame sebelumnya
        unmatched_ids = list(tracked_faces.keys())

        for i, (x, y, w, h) in enumerate(current_detections_bbox):
            center_x, center_y = x + w // 2, y + h // 2

            best_match_id = None
            min_dist_val = MAX_DISTANCE

            # Cari objek lama yang paling dekat
            for obj_id, data in tracked_faces.items():
                if obj_id not in unmatched_ids: continue # Sudah dicocokkan
                prev_bbox = data[0]
                prev_center_x, prev_center_y = prev_bbox[0] + prev_bbox[2] // 2, prev_bbox[1] + prev_bbox[3] // 2
                d = np.sqrt((center_x - prev_center_x)**2 + (center_y - prev_center_y)**2)
                if d < min_dist_val:
                    min_dist_val = d
                    best_match_id = obj_id

            if best_match_id is not None:
                # Ditemukan pasangan, update posisi dan tandai sebagai terlihat
                tracked_faces[best_match_id][0] = (x, y, w, h)
                tracked_faces[best_match_id][3] = frame_idx
                unmatched_ids.remove(best_match_id)
            else:
                # Objek baru
                tracked_faces[next_object_id] = [(x, y, w, h), "Processing...", 0.0, frame_idx]
                next_object_id += 1

        # Hapus objek yang sudah lama tidak terlihat
        for obj_id in list(tracked_faces.keys()):
            if frame_idx - tracked_faces[obj_id][3] > MAX_DISAPPEARED_FRAMES:
                del tracked_faces[obj_id]

        # Tahap 3: Lakukan Pengenalan dan Gambar Hasilnya
        for obj_id, data in tracked_faces.items():
            bbox, label, score, last_seen = data
            x1, y1, w, h = bbox
            x2, y2 = x1 + w, y1 + h

            # Lakukan pengenalan ulang jika label "Processing" atau sudah waktunya
            if label == "Processing..." or (frame_idx - last_seen) % RECOGNITION_INTERVAL == 0:
                # Gunakan crop ketat untuk pengenalan
                # ... (kode crop ketat Anda di sini) ...
                ih, iw, _ = frame.shape
                scale_factor = 0.15
                x_margin, y_margin = int(w * scale_factor), int(h * scale_factor)
                cx1, cy1 = x1 + x_margin, y1 + y_margin
                cx2, cy2 = x2 - x_margin, y2 - y_margin
                cx1, cy1, cx2, cy2 = max(0, cx1), max(0, cy1), min(iw, cx2), min(ih, cy2)

                if (cx2 - cx1) > 40 and (cy2 - cy1) > 40:
                    face_roi = frame[cy1:cy2, cx1:cx2]
                    if face_roi.size > 0:
                        preprocessed_face = preprocess_face_for_facenet(face_roi)
                        face_embedding_raw = embeddings_generator.predict(preprocessed_face, verbose=0)

                        raw_label, raw_score = recognize_face(
                            face_embedding_raw, known_face_embeddings_norm, known_face_labels,
                            best_classifier, normalizer, label_encoder,
                            prob_threshold=0.6, distance_threshold=1.1
                        )
                        if raw_label == "Unknown": raw_label = f"Unknown_{obj_id}"

                        # Perbarui ingatan tracker
                        tracked_faces[obj_id][1] = raw_label
                        tracked_faces[obj_id][2] = raw_score
                        label, score = raw_label, raw_score

            # === BLOK PENGGAMBARAN LENGKAP ===
            display_label = label.replace("_", " ") if "Unknown" not in label else f"Person {obj_id}"
            text_color = (0, 255, 0) if "Unknown" not in label else (0, 0, 255)

            # Gambar kotak deteksi
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), text_color, 2)

            # Siapkan dan gambar teks dengan latar belakang
            label_text = f"{display_label} ({score*100:.1f}%)"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.6
            font_thickness = 2
            (text_w, text_h), _ = cv2.getTextSize(label_text, font, font_scale, font_thickness)

            if y1 - text_h - 10 > 0:
                rect_start = (x1, y1 - text_h - 10)
                rect_end = (x1 + text_w + 6, y1 - 5)
                text_pos = (x1 + 3, y1 - 10)
            else:
                rect_start = (x1, y1 + 5)
                rect_end = (x1 + text_w + 6, y1 + text_h + 10)
                text_pos = (x1 + 3, y1 + text_h + 5)

            cv2.rectangle(annotated_frame, rect_start, rect_end, text_color, cv2.FILLED)
            cv2.putText(annotated_frame, label_text, text_pos, font, font_scale, (255, 255, 255), 1, cv2.LINE_AA)
            # === AKHIR BLOK PENGGAMBARAN ===

        out.write(annotated_frame)

    print("\nMelepaskan resource video...")
    cap.release()
    out.release()
    print(f"✅ Pemrosesan video selesai.")

In [ ]:

# --- Fungsi Preprocessing Wajah untuk FaceNet ---
def preprocess_face_for_facenet(face_img):
    """Preprocess gambar wajah yang terdeteksi untuk input FaceNet."""
    # Konversi BGR ke RGB jika berasal dari OpenCV
    if len(face_img.shape) == 3 and face_img.shape[2] == 3:
        face_img = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)

    # Ubah ukuran ke ukuran input FaceNet
    face_img = cv2.resize(face_img, (160, 160))
    face_img = face_img.astype('float32')

    # Normalisasi nilai piksel
    # FaceNet mengharapkan gambar dinormalisasi per gambar
    mean, std = face_img.mean(), face_img.std()
    # Hindari pembagian dengan nol jika standar deviasi nol
    if std == 0:
        img = face_img - mean
    else:
        img = (face_img - mean) / std

    return np.expand_dims(img, axis=0) # Tambahkan dimensi batch


# --- Fungsi Pengenalan Wajah ---
def recognize_face(embedding, known_embeddings, known_labels, classifier, normalizer, label_encoder, prob_threshold=0.7, distance_threshold=0.9):
    """
    Mengenali wajah berdasarkan embedding-nya.
    Menggunakan classifier yang terlatih terlebih dahulu, lalu pemeriksaan berbasis jarak untuk "Unknown".

    Args:
        embedding (np.array): Embedding 128-D dari wajah yang terdeteksi (belum dinormalisasi).
        known_embeddings (np.array): Array embedding wajah yang dikenal (sudah dinormalisasi).
        known_labels (np.array): Array label yang sesuai dengan known_embeddings.
        classifier: Objek classifier sklearn yang terlatih (SVC atau RandomForestClassifier).
        normalizer: Objek Normalizer sklearn.
        label_encoder: Objek LabelEncoder sklearn.
        prob_threshold (float): Ambang batas probabilitas untuk kepercayaan classifier.
        distance_threshold (float): Jarak L2 maksimum agar embedding dianggap 'known'.
                                     Nilai tipikal untuk embedding FaceNet yang dinormalisasi adalah sekitar ~0.8 hingga 1.2.

    Returns:
        tuple: (predicted_label: str, confidence_or_score: float)
    """
    embedding_norm = normalizer.transform(embedding.reshape(1, -1))

    # Prediksi dengan classifier yang terlatih
    probabilities = classifier.predict_proba(embedding_norm)[0]
    best_class_idx = np.argmax(probabilities)
    max_probability = probabilities[best_class_idx]
    predicted_label_from_classifier = label_encoder.inverse_transform([best_class_idx])[0]

    # Hitung jarak L2 ke semua embedding yang dikenal
    distances = np.linalg.norm(known_embeddings - embedding_norm, axis=1)
    min_distance = np.min(distances)

    # Logika Keputusan:
    # 1. Jika classifier cukup percaya diri, percaya pada prediksinya.
    if max_probability >= prob_threshold:
        return predicted_label_from_classifier, max_probability

    # 2. Jika classifier TIDAK percaya diri, tetapi embedding sangat dekat dengan wajah yang dikenal (berbasis jarak).
    # Ini berfungsi sebagai fallback untuk ketidakpastian classifier atau data yang noisy.
    elif min_distance < distance_threshold:
        idx_closest_known = np.argmin(distances)
        closest_known_label = known_labels[idx_closest_known]
        # Mengembalikan label dari wajah terdekat yang dikenal, dan "skor berbasis jarak" (1 - jarak_normalisasi)
        return closest_known_label, (1.0 - (min_distance / distance_threshold)) # Skala jarak ke skor 0-1

    # 3. Baik classifier tidak percaya diri maupun tidak ada kecocokan jarak yang dekat dengan wajah yang dikenal.
    else:
        return "Unknown", 0.0


#  Bab 7: Manajemen Unknown Face Database

In [ ]:


# --- Variabel Global untuk melacak wajah tidak dikenal ---
unknown_faces_db = {} # Menyimpan {unknown_id: raw_embedding}
unknown_counter = 0

def get_unique_unknown_id(new_embedding_raw, distance_threshold_for_unknown_tracking=0.8):
    """
    Memeriksa apakah embedding tidak dikenal yang baru mirip dengan yang sudah terdaftar.
    Jika mirip, mengembalikan ID yang sudah ada; jika tidak, menetapkan ID baru.
    """
    global unknown_counter, unknown_faces_db

    new_embedding_norm = normalizer.transform(new_embedding_raw) # Normalisasi untuk perbandingan

    for uid, stored_embedding_raw in unknown_faces_db.items():
        stored_embedding_norm = normalizer.transform(stored_embedding_raw)
        # Menggunakan np.linalg.norm pada array 1D akan menghasilkan skalar
        dist = np.linalg.norm(new_embedding_norm - stored_embedding_norm)
        if dist < distance_threshold_for_unknown_tracking: # Cek apakah itu orang tidak dikenal yang sama
            return uid

    # Jika tidak mirip dengan yang sudah ada, tetapkan ID baru
    unknown_counter += 1
    new_uid = f"Unknown_{unknown_counter}"
    unknown_faces_db[new_uid] = new_embedding_raw # Simpan embedding mentah
    print(f"✨ Terdeteksi orang tidak dikenal baru: {new_uid}")

    # Simpan database wajah tidak dikenal yang diperbarui
    save_unknown_faces_db()
    return new_uid

def save_unknown_faces_db():
    """Menyimpan unknown_faces_db saat ini ke file NPZ."""
    if unknown_faces_db:
        # Konversi dict ke list untuk disimpan
        unknown_labels_list = list(unknown_faces_db.keys())
        unknown_embeddings_list = list(unknown_faces_db.values())

        np.savez_compressed(os.path.join(output_files_path, 'unknown_persons_database.npz'),
                           embeddings=np.array(unknown_embeddings_list),
                           labels=np.array(unknown_labels_list))
        # print(f"💾 unknown_persons_database.npz diperbarui dengan {len(unknown_faces_db)} wajah tidak dikenal unik.") # Suppress frequent updates
    # else:
        # print("Tidak ada wajah tidak dikenal untuk disimpan.")

# Muat database wajah tidak dikenal yang sudah ada jika ada
if os.path.exists(os.path.join(output_files_path, 'unknown_persons_database.npz')):
    print("Memuat unknown_persons_database.npz yang sudah ada...")
    try:
        unknown_data = np.load(os.path.join(output_files_path, 'unknown_persons_database.npz'), allow_pickle=True)
        stored_embeddings = unknown_data['embeddings']
        stored_labels = unknown_data['labels']
        for i in range(len(stored_labels)):
            unknown_faces_db[stored_labels[i]] = stored_embeddings[i]

        # Perbarui unknown_counter berdasarkan ID unknown tertinggi yang ada
        max_unknown_id = 0
        for label in stored_labels:
            if label.startswith("Unknown_"):
                try:
                    num = int(label.split("_")[-1])
                    if num > max_unknown_id:
                        max_unknown_id = num
                except ValueError:
                    pass # Abaikan jika tidak dalam format yang diharapkan
        unknown_counter = max_unknown_id

        print(f"Dimuat {len(unknown_faces_db)} wajah tidak dikenal. ID unknown berikutnya dimulai dari {unknown_counter + 1}.")
    except Exception as e:
        print(f"❌ Error saat memuat unknown_persons_database.npz: {e}. Memulai dari awal.")
        unknown_faces_db = {}
        unknown_counter = 0


In [ ]:
# =========================================================================
# FUNGSI REGISTRASI (VERSI FINAL REVISI 2 - FIX NameError)
# =========================================================================

def register_person_multi_angle(person_name, image_paths, face_detector):
    """
    Mengekstrak embedding dari beberapa gambar seseorang, menggunakan MediaPipe untuk deteksi,
    menghitung rata-ratanya, dan mengembalikan satu embedding yang representatif.
    """
    all_embeddings = []
    print(f"🔍 Memproses registrasi untuk: {person_name} (menggunakan MediaPipe)")

    for img_path in image_paths:
        img = cv2.imread(img_path) # Gambar dimuat ke variabel 'img'
        if img is None:
            print(f"  ⚠️ Peringatan: Gagal memuat gambar {img_path}, dilewati.")
            continue

        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = face_detector.process(rgb_img)

        best_detection = None
        max_score = 0
        if results.detections:
            for detection in results.detections:
                if detection.score[0] > max_score:
                    max_score = detection.score[0]
                    best_detection = detection

        if best_detection is not None:
            bboxC = best_detection.location_data.relative_bounding_box

            # === PERBAIKAN ADA DI SINI ===
            # Kita menggunakan 'img.shape', bukan 'frame.shape'
            ih, iw, _ = img.shape
            # ============================

            # --- Logika Bounding Box yang lebih ketat ---
            x1_orig = int(bboxC.xmin * iw)
            y1_orig = int(bboxC.ymin * ih)
            w_orig = int(bboxC.width * iw)
            h_orig = int(bboxC.height * ih)

            scale_factor = 0.15
            x_margin = int(w_orig * scale_factor)
            y_margin = int(h_orig * scale_factor)

            x1 = x1_orig + x_margin
            y1 = y1_orig + y_margin
            x2 = (x1_orig + w_orig) - x_margin
            y2 = (y1_orig + h_orig) - y_margin

            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(iw, x2), min(ih, y2)

            face_roi = img[y1:y2, x1:x2]

            if face_roi.size > 0:
                preprocessed_face = preprocess_face_for_facenet(face_roi)
                embedding = embeddings_generator.predict(preprocessed_face, verbose=0)
                all_embeddings.append(embedding)
                print(f"  ✅ Ekstraksi embedding dari '{os.path.basename(img_path)}' berhasil.")
            else:
                print(f"  ❌ ROI wajah kosong untuk '{os.path.basename(img_path)}'.")
        else:
            print(f"  ❌ Tidak ada wajah terdeteksi di '{os.path.basename(img_path)}'.")

    if not all_embeddings:
        print(f"🚫 Gagal menghasilkan embedding untuk {person_name}. Tidak ada yang ditambahkan ke database.")
        return None, None

    stacked_embeddings = np.vstack(all_embeddings)
    average_embedding = np.mean(stacked_embeddings, axis=0)

    print(f"  ✨ Berhasil! Menghasilkan 1 embedding rata-rata dari {len(all_embeddings)} gambar untuk {person_name}.\n")

    return person_name, average_embedding.reshape(1, -1)

## versi registrasi json

In [ ]:
# =================================================================
# Bab 10 (Versi JSON): Membangun Database Wajah Terstruktur
# =================================================================
import json

print("🚀 Memulai pembangunan database wajah terstruktur (JSON)...")

# 1. Tentukan path ke data registrasi dan file output JSON
registration_base_path = os.path.join(output_files_path, 'registration_data')
database_json_path = os.path.join(output_files_path, 'face_database.json')

# 2. Temukan semua orang (sub-folder) di dalam folder registrasi
# Pastikan folder registration_data sudah ada
if not os.path.exists(registration_base_path):
    raise FileNotFoundError(f"Folder registrasi tidak ditemukan di: {registration_base_path}. Buat folder dan isi dengan foto terlebih dahulu.")

people_to_register = [d for d in os.listdir(registration_base_path) if os.path.isdir(os.path.join(registration_base_path, d))]

# List untuk menampung data setiap orang yang akan disimpan ke JSON
face_database = []

# 3. Loop melalui setiap orang dan daftarkan mereka
print(f"Ditemukan {len(people_to_register)} orang untuk didaftarkan: {people_to_register}")
for person_name in people_to_register:
    person_folder = os.path.join(registration_base_path, person_name)
    image_files = [os.path.join(person_folder, f) for f in os.listdir(person_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not image_files:
        print(f"⚠️ Peringatan: Tidak ada gambar ditemukan untuk {person_name}, dilewati.")
        continue

    # Panggil fungsi registrasi multi-angle yang sudah ada (tidak perlu diubah)
    # Fungsi ini mengembalikan label dan vektor rata-rata
    label, avg_embedding = register_person_multi_angle(person_name, image_files, face_detector)
    if label is not None and avg_embedding is not None:
        # Buat dictionary untuk menyimpan data orang ini secara terstruktur
        person_data = {
            "id": label,                           # ID unik, misal: "Rio_Aditya"
            "name": label.replace("_", " "),       # Nama yang lebih 'human-readable'
            "embedding": avg_embedding.flatten().tolist(), # PENTING: Konversi numpy array ke list agar valid di JSON
            "source_photos_count": len(image_files), # Metadata tambahan: berapa foto sumbernya
            "profile_photo_path": image_files[0]   # Simpan path foto pertama sebagai referensi 'foto profil'
        }
        face_database.append(person_data)

# 4. Simpan list of dictionaries ini sebagai satu file JSON
if face_database:
    # 'indent=4' membuat file JSON lebih mudah dibaca manusia
    with open(database_json_path, 'w') as f:
        json.dump(face_database, f, indent=4)

    print("\n========================================================")
    print(f"✅ Database wajah JSON telah dibuat dan disimpan di: {database_json_path}")
    print(f"Total orang berhasil terdaftar: {len(face_database)}")
    print("========================================================")
else:
    print("❌ Tidak ada orang yang berhasil didaftarkan. Database JSON tidak dibuat.")

🚀 Memulai pembangunan database wajah terstruktur (JSON)...
Ditemukan 4 orang untuk didaftarkan: ['rio_ganteng', 'Yande', 'Fathan', 'Icha']
🔍 Memproses registrasi untuk: rio_ganteng (menggunakan MediaPipe)
  ✅ Ekstraksi embedding dari 'WIN_20250711_05_48_34_Pro.jpg' berhasil.
  ✅ Ekstraksi embedding dari 'WIN_20250711_05_48_30_Pro.jpg' berhasil.
  ✅ Ekstraksi embedding dari 'WIN_20250711_05_48_27_Pro.jpg' berhasil.
  ✨ Berhasil! Menghasilkan 1 embedding rata-rata dari 3 gambar untuk rio_ganteng.

🔍 Memproses registrasi untuk: Yande (menggunakan MediaPipe)
  ✅ Ekstraksi embedding dari 'IMG-20250713-WA0066.jpg' berhasil.
  ✅ Ekstraksi embedding dari 'IMG-20250713-WA0067.jpg' berhasil.
  ✅ Ekstraksi embedding dari 'IMG-20250713-WA0068.jpg' berhasil.
  ✅ Ekstraksi embedding dari 'IMG-20250713-WA0069.jpg' berhasil.
  ✅ Ekstraksi embedding dari 'IMG-20250713-WA0070.jpg' berhasil.
  ✅ Ekstraksi embedding dari 'IMG-20250713-WA0071.jpg' berhasil.
  ✅ Ekstraksi embedding dari 'IMG-20250713-WA0072

# Eksekusi Inferensi pada File Video

In [ ]:
# =============================================================
# Bab 11: Eksekusi Inferensi pada File Video
# =============================================================
# Pastikan semua sel di atas sudah dijalankan dengan benar,
# terutama sel yang memuat database 'known_faces_multi_angle.npz'.

print("\n🚀 Memulai eksekusi inferensi pada file video...")

# 1. Pastikan video sumber sudah diupload ke Google Drive
# Ganti 'rio_test_video.mp4' dengan nama file video Anda jika berbeda
nama_file_video_input = 'tes_video.mp4'
input_video_path = os.path.join(output_files_path, nama_file_video_input)

# 2. Definisikan nama untuk file video hasil
nama_file_video_output = f"{os.path.splitext(nama_file_video_input)[0]}_hasil.mp4"
output_video_path = os.path.join(output_files_path, nama_file_video_output)

# 3. Periksa apakah file video input ada sebelum memulai
if not os.path.exists(input_video_path):
    print(f"❌ FATAL ERROR: File video input tidak ditemukan di '{input_video_path}'")
    print("Pastikan Anda sudah mengupload video ke folder 'Output Files' di Google Drive.")
else:
    # 4. Panggil fungsi untuk memproses video
    # Pastikan fungsi 'process_video_file' sudah didefinisikan di sel sebelumnya
    try:
        process_video_file(input_video_path, output_video_path)
        print("\n🎉 Inferensi video selesai! Periksa file output di Google Drive Anda.")
    except NameError:
        print("❌ FATAL ERROR: Fungsi 'process_video_file' tidak terdefinisi.")
        print("Pastikan Anda sudah menjalankan sel yang berisi definisi fungsi tersebut.")
    except Exception as e:
        print(f"❌ Terjadi error tak terduga selama pemrosesan video: {e}")


🚀 Memulai eksekusi inferensi pada file video...
🎬 Memproses video dengan Object Tracking: /content/drive/MyDrive/Magang KP Model AI /Output Files Facenet 512/tes_video.mp4


Memproses Frame: 100%|██████████| 397/397 [02:40<00:00,  2.47it/s]


Melepaskan resource video...
✅ Pemrosesan video selesai.

🎉 Inferensi video selesai! Periksa file output di Google Drive Anda.
